### Bitcoin

#### DLinear

In [6]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from autogluon.common import space

df = pd.read_csv("/home/yuyan/tabdpt_mz/data/source_data/bitcoin/bitcoin_price_news_gpt52_concise_summary.csv")
df = df.sort_values("Date").reset_index(drop=True)[['Date', 'Open',	'High',	'Low',	'Close','Volume']]	

# --- Prepare ordered data ---
df = df.sort_values("Date").reset_index(drop=True)
df_1000 = df.iloc[:1000].copy()
df_1000["item_id"] = 1

train_df = df_1000.iloc[:700]
val_df   = df_1000.iloc[700:850]
test_df  = df_1000.iloc[850:1000]

train_data = TimeSeriesDataFrame.from_data_frame(train_df, id_column="item_id", timestamp_column="Date")
val_data   = TimeSeriesDataFrame.from_data_frame(val_df,   id_column="item_id", timestamp_column="Date")
test_data  = TimeSeriesDataFrame.from_data_frame(test_df,  id_column="item_id", timestamp_column="Date")

# --- Step 1: HPO on train + val ---
predictor = TimeSeriesPredictor(
    target="Close",
    prediction_length=1,
    eval_metric="MAE",
)

predictor.fit(
    train_data,
    tuning_data=val_data,
    hyperparameters={
        "DLinear": {
            "context_length": space.Int(2, 20),
            "hidden_dimension": space.Int(8, 64),
            "kernel_size": space.Int(3, 25),
            "lr": space.Real(1e-4, 3e-3, log=True),
            "batch_size": space.Categorical(32, 64, 128),
            "max_epochs": space.Int(20, 200),
        }
    },
    hyperparameter_tune_kwargs={
        "num_trials": 20,
        "searcher": "random",
        "scheduler": "local",
    },
    enable_ensemble=False,
)

# Grab best HP
summary = predictor.fit_summary()
best_model = summary["model_best"]
best_hps = summary["model_hyperparams"][best_model]

print("Best model:", best_model)
print("Best hps:", best_hps)

# --- Step 2: walk-forward refit on train+val then predict each test point ---
# Build full history up to start of test
history_df = df_1000.iloc[:850].copy()

preds = []
for i in range(850, 1000):
    # Train data up to the point we want to predict (inclusive)
    hist_df = df_1000.iloc[:i].copy()  # up to i-1 index (1..i in 1-based)
    hist_df["item_id"] = 1
    hist_ts = TimeSeriesDataFrame.from_data_frame(hist_df, id_column="item_id", timestamp_column="Date")

    # Refit with best hyperparams
    wf_predictor = TimeSeriesPredictor(
        target="Close",
        prediction_length=1,
        eval_metric="MAE",
        path=f"ag_walkforward_{i}",  # optional; keep separate model dirs
    )
    wf_predictor.fit(
        hist_ts,
        hyperparameters={"DLinear": best_hps},
        enable_ensemble=False,
    )

    # Predict next point (i+1 in 1-based, i index in 0-based)
    fcst = wf_predictor.predict(hist_ts)
    pred_value = fcst["mean"].iloc[-1]
    true_value = df_1000.loc[i, "Close"]

    preds.append({"index": i, "date": df_1000.loc[i, "Date"], "y_true": true_value, "y_pred": pred_value})

preds_df = pd.DataFrame(preds)
print(preds_df.head())



	Trained 10 models while tuning DLinear.
	-977.1577     = Validation score (-MAE)
	93.23   s     = Total tuning time
Training complete. Models trained: ['DLinear/fe140_00000', 'DLinear/fe140_00001', 'DLinear/fe140_00004', 'DLinear/fe140_00005', 'DLinear/fe140_00007', 'DLinear/fe140_00008', 'DLinear/fe140_00011', 'DLinear/fe140_00016', 'DLinear/fe140_00017', 'DLinear/fe140_00019']
Total runtime: 93.29 s
Best model: DLinear/fe140_00004
Best model score: -977.1577
Beginning AutoGluon training...
AutoGluon will save models to '/home/yuyan/tabdpt_mz/notebooks/ag_walkforward_850'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #91~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Nov 20 15:20:45 UTC 2
CPU Count:          112
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 47.48/47.49 GB | GPU 1: 47.51/47.51 GB
Total GPU Memory:   Free: 94

****************** Summary of fit() ******************
Estimated performance of each model:
                 model    score_val  pred_time_val  fit_time_marginal  \
0  DLinear/fe140_00004  -977.157676       0.179922           9.698470   
1  DLinear/fe140_00016  -981.182090       0.177118           8.103105   
2  DLinear/fe140_00007  -985.926719       0.175323          10.481062   
3  DLinear/fe140_00005 -1186.394980       0.183650           8.921923   
4  DLinear/fe140_00011 -1215.800742       0.177275          10.649420   
5  DLinear/fe140_00001 -1261.982871       0.189540           9.225395   
6  DLinear/fe140_00008 -1265.565391       0.182516           9.084224   
7  DLinear/fe140_00000 -1311.654746       0.179655           7.228399   
8  DLinear/fe140_00017 -1358.457969       0.223757           8.832788   
9  DLinear/fe140_00019 -1476.640098       0.172971           7.764290   

   fit_order  
0          3  
1          8  
2          5  
3          4  
4          7  
5          2  

	-972.2334     = Validation score (-MAE)
	6.39    s     = Training runtime
	0.01    s     = Validation (prediction) runtime
Training complete. Models trained: ['DLinear']
Total runtime: 6.42 s
Best model: DLinear
Best model score: -972.2334
Model not specified in predict, will default to the model with the best validation score: DLinear
Beginning AutoGluon training...
AutoGluon will save models to '/home/yuyan/tabdpt_mz/notebooks/ag_walkforward_851'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #91~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Nov 20 15:20:45 UTC 2
CPU Count:          112
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 47.48/47.49 GB | GPU 1: 47.51/47.51 GB
Total GPU Memory:   Free: 94.98 GB, Allocated: 0.01 GB, Total: 94.99 GB
GPU Count:          2
Memory Avail:       992.02 GB / 1006.49 GB (98.6%)
Disk Space

   index        date   y_true       y_pred
0    850  2020-04-30  8658.55  8620.243164
1    851  2020-05-01  8864.77  8646.345703
2    852  2020-05-02  8988.60  8912.178711
3    853  2020-05-03  8897.47  9086.539062
4    854  2020-05-04  8912.65  8871.918945


In [7]:
import numpy as np

err = preds_df["y_true"] - preds_df["y_pred"]

mae = err.abs().mean()
rmse = np.sqrt(np.mean(err ** 2))

# Avoid divide-by-zero in MAPE
eps = 1e-8
mape = np.mean(np.abs(err) / (np.abs(preds_df["y_true"]) + eps)) * 100

print("MAE:", mae)

print("RMSE:", rmse)
print("MAPE (%):", mape)

MAE: 203.85732031249998
RMSE: 281.4170483434349
MAPE (%): 2.0075503608691894


#### Chronos

In [8]:
# chronos bitcoin
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
df = pd.read_csv("/home/yuyan/tabdpt_mz/data/source_data/bitcoin/bitcoin_price_news_gpt52_concise_summary.csv")
df = df.sort_values("Date").reset_index(drop=True)[['Date', 'Open',	'High',	'Low',	'Close','Volume']]	

# df_1000 already sorted by Date, item_id set, and has Close column
df_1000 = df.sort_values("Date").reset_index(drop=True).iloc[:1000].copy()
df_1000["item_id"] = 1

# Chronos predictor (no HPO needed for zero-shot)
predictor = TimeSeriesPredictor(
    target="Close",
    prediction_length=1,
)
predictor.fit(
    TimeSeriesDataFrame.from_data_frame(
        df_1000.iloc[:700], id_column="item_id", timestamp_column="Date"
    ),
    hyperparameters={
        "Chronos2": {"model_path": "autogluon/chronos-2"}
    },
    enable_ensemble=False,
)

# Rolling window prediction on test indices 850..999 (1-based 851..1000)
preds = []
for i in range(850, 1000):
    hist_df = df_1000.iloc[:i].copy()  # context 1..i (1-based)
    hist_ts = TimeSeriesDataFrame.from_data_frame(
        hist_df, id_column="item_id", timestamp_column="Date"
    )

    fcst = predictor.predict(hist_ts)
    y_pred = fcst["mean"].iloc[-1]
    y_true = df_1000.loc[i, "Close"]

    preds.append({"index": i, "date": df_1000.loc[i, "Date"], "y_true": y_true, "y_pred": y_pred})

preds_df = pd.DataFrame(preds)


No path specified. Models will be saved in: "AutogluonModels/ag-20260209_043623"
Beginning AutoGluon training...
AutoGluon will save models to '/home/yuyan/tabdpt_mz/notebooks/AutogluonModels/ag-20260209_043623'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #91~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Nov 20 15:20:45 UTC 2
CPU Count:          112
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 47.48/47.49 GB | GPU 1: 47.51/47.51 GB
Total GPU Memory:   Free: 94.98 GB, Allocated: 0.01 GB, Total: 94.99 GB
GPU Count:          2
Memory Avail:       992.20 GB / 1006.49 GB (98.6%)
Disk Space Avail:   206.91 GB / 925.10 GB (22.4%)

Fitting with arguments:
{'enable_ensemble': False,
 'eval_metric': WQL,
 'hyperparameters': {'Chronos2': {'model_path': 'autogluon/chronos-2'}},
 'known_covariates_names': [],
 'num_val_windows': 1,
 '

In [9]:
import numpy as np

err = preds_df["y_true"] - preds_df["y_pred"]

mae = err.abs().mean()
rmse = np.sqrt(np.mean(err ** 2))

# Avoid divide-by-zero in MAPE
eps = 1e-8
mape = np.mean(np.abs(err) / (np.abs(preds_df["y_true"]) + eps)) * 100

print("MAE:", mae)

print("RMSE:", rmse)
print("MAPE (%):", mape)

MAE: 195.8591432291667
RMSE: 280.0870610818837
MAPE (%): 1.9211018607544612


### Energy

#### DLinear

In [ ]:
# energy
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from autogluon.common import space

# Load energy data
df = pd.read_csv("/home/yuyan/tabdpt_mz/data/source_data/energy/Energy_with_text_summarized.csv")

# Keep only date + numeric columns (drop text)
df = df.sort_values("date").reset_index(drop=True)
numeric_cols = df.select_dtypes(include=["number"]).columns
df = df[["date"] + list(numeric_cols)]

target_col = "OT"  # change if you want a different target

# Slice first 1000 points and make single series
df_1000 = df.iloc[:1000].copy()
df_1000["item_id"] = 1

train_df = df_1000.iloc[:700]
val_df   = df_1000.iloc[700:850]
test_df  = df_1000.iloc[850:1000]

train_data = TimeSeriesDataFrame.from_data_frame(train_df, id_column="item_id", timestamp_column="date")
val_data   = TimeSeriesDataFrame.from_data_frame(val_df,   id_column="item_id", timestamp_column="date")
test_data  = TimeSeriesDataFrame.from_data_frame(test_df,  id_column="item_id", timestamp_column="date")

# --- Step 1: HPO on train + val ---
predictor = TimeSeriesPredictor(
    target=target_col,
    prediction_length=1,
    eval_metric="MAE",
)

predictor.fit(
    train_data,
    tuning_data=val_data,
    hyperparameters={
        "DLinear": {
            "context_length": space.Int(2, 20),
            "hidden_dimension": space.Int(8, 64),
            "kernel_size": space.Int(3, 25),
            "lr": space.Real(1e-4, 3e-3, log=True),
            "batch_size": space.Categorical(32, 64, 128),
            "max_epochs": space.Int(20, 200),
        }
    },
    hyperparameter_tune_kwargs={
        "num_trials": 20,
        "searcher": "random",
        "scheduler": "local",
    },
    enable_ensemble=False,
)

summary = predictor.fit_summary()
best_model = summary["model_best"]
best_hps = summary["model_hyperparams"][best_model]
print("Best model:", best_model)
print("Best hps:", best_hps)

# --- Step 2: walk-forward refit + predict ---
preds = []
for i in range(850, 1000):
    hist_df = df_1000.iloc[:i].copy()
    hist_df["item_id"] = 1
    hist_ts = TimeSeriesDataFrame.from_data_frame(hist_df, id_column="item_id", timestamp_column="date")

    wf_predictor = TimeSeriesPredictor(
        target=target_col,
        prediction_length=1,
        eval_metric="MAE",
        path=f"ag_walkforward_energy_{i}",
    )
    wf_predictor.fit(
        hist_ts,
        hyperparameters={"DLinear": best_hps},
        enable_ensemble=False,
    )

    fcst = wf_predictor.predict(hist_ts)
    y_pred = fcst["mean"].iloc[-1]
    y_true = df_1000.loc[i, target_col]

    preds.append({"index": i, "date": df_1000.loc[i, "date"], "y_true": y_true, "y_pred": y_pred})

preds_df = pd.DataFrame(preds)



	Trained 10 models while tuning DLinear.
	-0.0018       = Validation score (-MAE)
	120.88  s     = Total tuning time
Training complete. Models trained: ['DLinear/045fa_00000', 'DLinear/045fa_00001', 'DLinear/045fa_00004', 'DLinear/045fa_00005', 'DLinear/045fa_00007', 'DLinear/045fa_00008', 'DLinear/045fa_00011', 'DLinear/045fa_00016', 'DLinear/045fa_00017', 'DLinear/045fa_00019']
Total runtime: 120.95 s
Best model: DLinear/045fa_00008
Best model score: -0.0018
Beginning AutoGluon training...
AutoGluon will save models to '/home/yuyan/tabdpt_mz/notebooks/ag_walkforward_energy_850'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #91~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Nov 20 15:20:45 UTC 2
CPU Count:          112
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 47.48/47.49 GB | GPU 1: 47.51/47.51 GB
Total GPU Memory:   Fr

****************** Summary of fit() ******************
Estimated performance of each model:
                 model  score_val  pred_time_val  fit_time_marginal  fit_order
0  DLinear/045fa_00008  -0.001808       0.173532          22.489501          6
1  DLinear/045fa_00016  -0.006291       0.174587          11.988254          8
2  DLinear/045fa_00017  -0.014418       0.178359          19.608154          9
3  DLinear/045fa_00001  -0.021127       0.172025          16.357142          2
4  DLinear/045fa_00019  -0.022512       0.171883           7.834596         10
5  DLinear/045fa_00011  -0.029610       0.190255          15.770522          7
6  DLinear/045fa_00005  -0.029691       0.174249          13.903959          4
7  DLinear/045fa_00004  -0.041082       0.181093          19.100434          3
8  DLinear/045fa_00007  -0.048205       0.174883          16.350090          5
9  DLinear/045fa_00000  -0.162112       0.175460           7.215643          1
Number of models trained: 10
Types of m

	-0.0003       = Validation score (-MAE)
	9.79    s     = Training runtime
	0.01    s     = Validation (prediction) runtime
Training complete. Models trained: ['DLinear']
Total runtime: 9.82 s
Best model: DLinear
Best model score: -0.0003
Model not specified in predict, will default to the model with the best validation score: DLinear
Beginning AutoGluon training...
AutoGluon will save models to '/home/yuyan/tabdpt_mz/notebooks/ag_walkforward_energy_851'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #91~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Nov 20 15:20:45 UTC 2
CPU Count:          112
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 47.48/47.49 GB | GPU 1: 47.51/47.51 GB
Total GPU Memory:   Free: 94.98 GB, Allocated: 0.01 GB, Total: 94.99 GB
GPU Count:          2
Memory Avail:       991.96 GB / 1006.49 GB (98.6%)
Disk 

#### Chronos

In [ ]:
import numpy as np

err = preds_df["y_true"] - preds_df["y_pred"]

mae = err.abs().mean()
rmse = np.sqrt(np.mean(err ** 2))

# Avoid divide-by-zero in MAPE
eps = 1e-8
mape = np.mean(np.abs(err) / (np.abs(preds_df["y_true"]) + eps)) * 100

print("MAE:", mae)

print("RMSE:", rmse)
print("MAPE (%):", mape)


MAE: 203.85732031249998
RMSE: 281.4170483434349
MAPE (%): 2.0075503608691894


In [ ]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

df = pd.read_csv("/home/yuyan/tabdpt_mz/data/source_data/energy/Energy_with_text_summarized.csv")
df = df.sort_values("date").reset_index(drop=True)

# keep date + numeric columns (drop text)
numeric_cols = df.select_dtypes(include=["number"]).columns
df = df[["date"] + list(numeric_cols)]

target_col = "OT"

df_1000 = df.iloc[:1000].copy()
df_1000["item_id"] = 1

# fit once on the initial train segment (1..700)
predictor = TimeSeriesPredictor(
    target=target_col,
    prediction_length=1,
)

predictor.fit(
    TimeSeriesDataFrame.from_data_frame(
        df_1000.iloc[:700], id_column="item_id", timestamp_column="date"
    ),
    hyperparameters={        "Chronos2": {"model_path": "autogluon/chronos-2"}
},  # adjust if you use Chronos-2 name
    enable_ensemble=False,
)

# rolling-context predictions for 851..1000
preds = []
for i in range(850, 1000):
    hist_df = df_1000.iloc[:i].copy()
    hist_ts = TimeSeriesDataFrame.from_data_frame(
        hist_df, id_column="item_id", timestamp_column="date"
    )

    fcst = predictor.predict(hist_ts)
    y_pred = fcst["mean"].iloc[-1]
    y_true = df_1000.loc[i, target_col]

    preds.append({"index": i, "date": df_1000.loc[i, "date"], "y_true": y_true, "y_pred": y_pred})

preds_df = pd.DataFrame(preds)
# mae = (preds_df["y_true"] - preds_df["y_pred"]).abs().mean()
# print("Chronos rolling MAE (energy):", mae)


No path specified. Models will be saved in: "AutogluonModels/ag-20260209_025754"
Beginning AutoGluon training...
AutoGluon will save models to '/home/yuyan/tabdpt_mz/notebooks/AutogluonModels/ag-20260209_025754'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #91~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Nov 20 15:20:45 UTC 2
CPU Count:          112
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 47.48/47.49 GB | GPU 1: 47.51/47.51 GB
Total GPU Memory:   Free: 94.98 GB, Allocated: 0.01 GB, Total: 94.99 GB
GPU Count:          2
Memory Avail:       992.34 GB / 1006.49 GB (98.6%)
Disk Space Avail:   207.67 GB / 925.10 GB (22.4%)

Fitting with arguments:
{'enable_ensemble': False,
 'eval_metric': WQL,
 'hyperparameters': {'Chronos2': {'model_path': 'autogluon/chronos-2'}},
 'known_covariates_names': [],
 'num_val_windows': 1,
 '

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

	-0.0141       = Validation score (-WQL)
	5.46    s     = Training runtime
	0.11    s     = Validation (prediction) runtime
Training complete. Models trained: ['Chronos2']
Total runtime: 5.58 s
Best model: Chronos2
Best model score: -0.0141
Model not specified in predict, will default to the model with the best validation score: Chronos2
Model not specified in predict, will default to the model with the best validation score: Chronos2
Model not specified in predict, will default to the model with the best validation score: Chronos2
Model not specified in predict, will default to the model with the best validation score: Chronos2
Model not specified in predict, will default to the model with the best validation score: Chronos2
Model not specified in predict, will default to the model with the best validation score: Chronos2
Model not specified in predict, will default to the model with the best validation score: Chronos2
Model not specified in predict, will default to the model with the

Chronos rolling MAE (energy): 0.04629375047047933


In [ ]:
import numpy as np
err = preds_df["y_true"] - preds_df["y_pred"]

mae = err.abs().mean()
rmse = np.sqrt(np.mean(err ** 2))

# Avoid divide-by-zero in MAPE
eps = 1e-8
mape = np.mean(np.abs(err) / (np.abs(preds_df["y_true"]) + eps)) * 100

print("MAE:", mae)

print("RMSE:", rmse)
print("MAPE (%):", mape)